In [1]:
from pathlib import Path
from datetime import date
import json
import sys

import pandas as pd

sys.path.append(r"C:\Users\F.Turner\Documents\00. Analyses")
from use_funcs import find_project_root

import api_helpers.imf_api as imf_api
from api_helpers.uis_api import *



# 01. Government Expenditure by Sector

This notebook uses a config file (`path_config.json`) so file paths are not hardcoded.

In [2]:
PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "path_config.json"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PATHS = json.load(f)

IMP_DIR = (PROJECT_ROOT / PATHS["imports_dir"]).resolve()
HO_DIR = (PROJECT_ROOT / PATHS["handoff_dir"]).resolve()
EXP_DIR = (PROJECT_ROOT / PATHS["exports_dir"]).resolve()
FIG_DIR = (PROJECT_ROOT / PATHS["figures_dir"]).resolve()
TAB_DIR = (PROJECT_ROOT / PATHS["tables_dir"]).resolve()

for folder in [IMP_DIR, HO_DIR, EXP_DIR, FIG_DIR, TAB_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMP_DIR:", IMP_DIR)
print("HO_DIR:", HO_DIR)
print("EXP_DIR:", EXP_DIR)
print("FIG_DIR:", FIG_DIR)
print("TAB_DIR:", TAB_DIR)

PROJECT_ROOT: C:\Users\F.Turner\Documents\00. Analyses\Education Financing
IMP_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Data
HO_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Handoff
EXP_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results
FIG_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results\figures
TAB_DIR: C:\Users\F.Turner\Documents\00. Analyses\Education Financing\Results\tables


In [3]:
API_KEY = "51addd610c2d4725806a8a49c1eaab8f"

# **MODEL 1: Common Functions of Government (IMF Data)**

## IMF Pull (api_helpers)

Use `imf_get_gfs_cofog()` to download COFOG expenditure shares (`POTO_PT`) in one call.

In [4]:
import importlib

imf_api = importlib.reload(imf_api)

START_YEAR = 2000
END_YEAR = 2026

gfs_df = imf_api.imf_get_gfs_cofog(
    transformations="POTO_PT",
    frequency="A",
    start_year=START_YEAR,
    end_year=END_YEAR,
    format="long",
    api_key=API_KEY,
)

print(f"Downloaded {len(gfs_df):,} observations")
print(f"Period: {START_YEAR} - {END_YEAR}")
display(gfs_df.head())

Downloaded 506,332 observations
Period: 2000 - 2026


,COUNTRY,COUNTRY_label,SECTOR,SECTOR_label,GFS_GRP,GFS_GRP_label,INDICATOR,INDICATOR_label,TYPE_OF_TRANSFORMATION,TYPE_OF_TRANSFORMATION_label,FREQUENCY,FREQUENCY_label,TIME_PERIOD,TIME_PERIOD_label,value
0,AFG,"Afghanistan, Islamic Republic of",S13,General government,G2MF,Expenditure by function,GF011_T,"Executive and legislative organs, financial an...",POTO_PT,Percent of total outlays,A,Annual,2016,2016,6.18639026089096
1,AFG,"Afghanistan, Islamic Republic of",S13,General government,G2MF,Expenditure by function,GF011_T,"Executive and legislative organs, financial an...",POTO_PT,Percent of total outlays,A,Annual,2017,2017,5.17067317090786
2,AFG,"Afghanistan, Islamic Republic of",S13,General government,G2MF,Expenditure by function,GF012_T,"Foreign economic aid, Transactions",POTO_PT,Percent of total outlays,A,Annual,2016,2016,0
3,AFG,"Afghanistan, Islamic Republic of",S13,General government,G2MF,Expenditure by function,GF012_T,"Foreign economic aid, Transactions",POTO_PT,Percent of total outlays,A,Annual,2017,2017,0
4,AFG,"Afghanistan, Islamic Republic of",S13,General government,G2MF,Expenditure by function,GF013_T,"General services, Transactions",POTO_PT,Percent of total outlays,A,Annual,2016,2016,0.294639680164992


## Build Table 1: All Sectors (GF01_T to GF10_T)

Create one country-year table with all aggregate COFOG sector shares.

In [5]:
agg_src = gfs_df[gfs_df["INDICATOR"].str.match(r"^GF(0[1-9]|10)_T$", na=False)].copy()
agg_src["TIME_PERIOD"] = pd.to_numeric(agg_src["TIME_PERIOD"], errors="coerce")
agg_src["value"] = pd.to_numeric(agg_src["value"], errors="coerce")
agg_src = agg_src.dropna(subset=["TIME_PERIOD", "value"]).copy()
agg_src["TIME_PERIOD"] = agg_src["TIME_PERIOD"].astype(int)

agg_wide = (
    agg_src.pivot_table(
        index=["COUNTRY", "COUNTRY_label", "TIME_PERIOD"],
        columns="INDICATOR",
        values="value",
        aggfunc="first",
    )
    .reset_index()
    .sort_values(["COUNTRY", "TIME_PERIOD"])
)

sector_codes = [f"GF{i:02d}_T" for i in range(1, 11)]
agg_wide = agg_wide.dropna(subset=sector_codes).copy()
agg_wide[sector_codes] = agg_wide[sector_codes].div(100, axis=0)
agg_wide["sector_sum_pct"] = agg_wide[sector_codes].sum(axis=1)

label_map = (
    agg_src[["INDICATOR", "INDICATOR_label"]]
    .drop_duplicates()
    .set_index("INDICATOR")["INDICATOR_label"]
    .to_dict()
)
agg_wide = agg_wide.rename(columns=label_map)

In [6]:
agg_wide = agg_wide.drop(columns=['COUNTRY_label'])

agg_wide = agg_wide.rename(columns={"COUNTRY": "iso3", 
                                    "TIME_PERIOD": "year",
                                    'General public services, Transactions': 'general_public_services_pct',
                                    'Defence, Transactions': 'defense_pct',
                                    'Public order and safety, Transactions': 'public_order_safety_pct',
                                    'Economic affairs, Transactions': 'economic_affairs_pct',
                                    'Environmental protection, Transactions': 'environmental_protection_pct',
                                    'Housing and community amenities, Transactions': 'housing_community_amenities_pct',
                                    'Health, Transactions': 'health_pct',
                                    'Recreation, culture and religion, Transactions': 'recreation_culture_religion_pct',
                                    'Education, Transactions': 'education_pct',
                                    'Social protection, Transactions': 'social_protection_pct'
                                    })

In [7]:
aqq_wide = agg_wide.drop(columns=['COUNTRY_label'])

KeyError: "['COUNTRY_label'] not found in axis"

In [ ]:
agg_wide.to_csv(IMP_DIR / "cofog_all.csv", index=False)

In [ ]:
agg_wide.columns

Index(['iso3', 'year', 'general_public_services_pct', 'defense_pct',
       'public_order_safety_pct', 'economic_affairs_pct',
       'environmental_protection_pct', 'housing_community_amenities_pct',
       'health_pct', 'recreation_culture_religion_pct', 'education_pct',
       'social_protection_pct', 'sector_sum_pct'],
      dtype='str', name='INDICATOR')

## Build Table 2: Education Total and Sub-sectors (GF09 family)

Create one country-year table with education total and education sub-sector shares.

In [8]:
edu_src = gfs_df[gfs_df["INDICATOR"].str.match(r"^GF09\d*_T$", na=False)].copy()
edu_src["TIME_PERIOD"] = pd.to_numeric(edu_src["TIME_PERIOD"], errors="coerce")
edu_src["value"] = pd.to_numeric(edu_src["value"], errors="coerce")
edu_src = edu_src.dropna(subset=["TIME_PERIOD", "value"]).copy()
edu_src["TIME_PERIOD"] = edu_src["TIME_PERIOD"].astype(int)

gfs_country_year_education = (
    edu_src.pivot_table(
        index=["COUNTRY", "COUNTRY_label", "TIME_PERIOD"],
        columns="INDICATOR_label",
        values="value",
        aggfunc="first",
    )
    .reset_index()
    .sort_values(["COUNTRY", "TIME_PERIOD"])
)

gfs_country_year_education.columns = [str(c) for c in gfs_country_year_education.columns]
gfs_country_year_education.columns.name = None

# Identify subsector columns (all indicator columns, excluding the GF09_T aggregate).
id_cols = ["COUNTRY", "COUNTRY_label", "TIME_PERIOD"]
indicator_cols = [c for c in gfs_country_year_education.columns if c not in id_cols]

# Find the education total column name (GF09_T) for use as denominator.
edu_total_col = next(
    (c for c in indicator_cols if "Education, Transactions" in c or c == "Education"),
    None,
)

# Normalise all indicator values as a share of the education total (GF09_T).
# Rows where the total is zero or missing are left as NaN.
if edu_total_col:
    denom = gfs_country_year_education[edu_total_col].replace(0, float("nan"))
    gfs_country_year_education[indicator_cols] = (
        gfs_country_year_education[indicator_cols].div(denom, axis=0)
    )


In [9]:
gfs_country_year_education = gfs_country_year_education.drop(columns=['COUNTRY_label'])

gfs_country_year_education = gfs_country_year_education.rename(columns={"COUNTRY": "iso3",
                                                                    "TIME_PERIOD": "year",
                                                                    'Education, Transactions': 'education_total_pct',
                                                                    'Pre-primary and primary education, Transactions': 'ecd_and_primary_pct',
                                                                    'Secondary education, Transactions': 'secondary_pct',
                                                                    'Tertiary education, Transactions': 'tertiary_pct',
                                                                    'Education not definable by level, Transactions': 'education_not_definable_by_level_pct',
                                                                    'Education, Transactions': 'education_total_pct',
                                                                    'Education n.e.c., Transactions': 'education_nec_pct',
                                                                    'R&D Education, Transactions': 'education_rnd_pct',
                                                                    'Subsidiary services to education, Transactions': 'education_subsidiary_services_pct',
                                                                    'Post-secondary non-tertiary education, Transactions': 'post_secondary_non_tertiary_pct'
                                                                    })


In [ ]:
gfs_country_year_education.to_csv(IMP_DIR / "cofog_edu.csv", index=False)

# **MODEL 2: Recurrent Spending v Capital Investment**

In [58]:
df = uis_search_indicators("per primary student")

df

,indicator_id,indicator_name,theme,last_data_update,last_data_update_description,total_record_count,timeline_min,timeline_max,entity_types
0,XUNIT.GDPCAP.1.FSGOV.FFNTR,Initial government funding per primary student...,EDUCATION,2026-02-09,February 2026 Data Release,1971,1997,2024,[NATIONAL]
1,XUNIT.PPPCONST.1.FSGOV.FFNTR,Initial government funding per primary student...,EDUCATION,2026-02-09,February 2026 Data Release,1930,1997,2024,[NATIONAL]
2,XUNIT.GDPCAP.1.FSHH.FFNTR,Initial household funding per primary student ...,EDUCATION,2026-02-09,February 2026 Data Release,802,1998,2024,[NATIONAL]
3,XUNIT.PPPCONST.1.FSHH.FFNTR,"Initial household funding per primary student,...",EDUCATION,2026-02-09,February 2026 Data Release,791,1998,2024,[NATIONAL]


In [60]:

cap_recu = uis_get(indicators=['XGOVEXP.IMFCALC',
                               'XUNIT.PPPCONST.1.FSGOV.FFNTR',
                               'FHLANGILP.PRIMARY',
                               'XGOVEXP.IMF',
                               'XGDP.FSGOV',
                               'XSPENDP.FDPUB.FNS', 
                               'XSPENDP.FDPUB.FNCAP', 
                               'XSPENDP.FDPUB.FNCUR', 
                               'XSPENDP.FDPUB.FNTS'], 
        start_year=2010, 
        end_year = 2026)

In [62]:
cap_recu = cap_recu.rename(columns={"entity_id":'iso3'})
cap_recu = cap_recu.drop(columns=['magnitude', 'qualifier'])

map = {'XGOVEXP.IMFCALC':'expenditure_pctbudget_imf', 
       'XGOVEXP.IMF': 'expenditure_pctbudget_uis',
       'XGDP.FSGOV':'expenditure_pctgdp',
       'XUNIT.PPPCONST.1.FSGOV.FFNTR':'expenditure_perchild_ppp',      
       'XSPENDP.FDPUB.FNCAP':'capex_pcttotal',
       'XSPENDP.FDPUB.FNCUR' : 'currex_pcttotal',
       'XSPENDP.FDPUB.FNS': 'all_staff_pcttotal',
       'XSPENDP.FDPUB.FNTS': 'teaching_staff_pcttotal',
       'FHLANGILP.PRIMARY': 'lang_match_pct'}

cap_recu['indicator_name'] = cap_recu['indicator_id'].map(map)
cap_recu = cap_recu[['iso3', 'indicator_name','indicator_id', 'year', 'value']]

cap_recu['value'] = cap_recu['value'] / 100

In [63]:
cap_recu_wide = pd.pivot_table(
    cap_recu,
    index=['iso3', 'year'],
    columns='indicator_name',
    values='value'
).reset_index().rename_axis(columns=None)

In [65]:
cap_recu_wide['expenditure_perchild_ppp'] = cap_recu_wide['expenditure_perchild_ppp'] * 100
cap_recu_wide.describe()

,year,all_staff_pcttotal,capex_pcttotal,currex_pcttotal,expenditure_pctbudget_imf,expenditure_pctbudget_uis,expenditure_pctgdp,expenditure_perchild_ppp,lang_match_pct,teaching_staff_pcttotal
count,3303.000000,1051.000000,1108.000000,1115.000000,1326.000000,2941.000000,2.273000e+03,1123.000000,137.000000,675.000000
mean,2017.029670,0.690957,0.089443,0.910700,0.137357,0.142431,4.359848e-02,5931.090457,0.659197,0.567970
std,4.299476,0.126685,0.074416,0.076077,0.051289,0.045818,1.970897e-02,5802.135848,0.327165,0.138389
min,2010.000000,0.000000,0.000000,0.328100,0.014444,0.000000,4.009908e-08,3.086040,0.015105,0.000000
25%,2013.000000,0.634027,0.042542,0.885029,0.101214,0.113104,3.077620e-02,1437.890808,0.392520,0.471320
50%,2017.000000,0.699795,0.074354,0.926011,0.128977,0.139423,4.178460e-02,3830.977539,0.803494,0.577948
75%,2021.000000,0.759931,0.115674,0.957979,0.166217,0.166209,5.366940e-02,9291.694824,0.923172,0.658408
max,2025.000000,1.000000,0.671900,1.000000,0.377008,0.350100,1.639053e-01,34286.859375,0.992236,1.000000


In [66]:
cap_recu_wide.to_csv(IMP_DIR / "expenditure.csv", index=False)